# Exploratory Data Analysis: Automobile Insurance Fraud

This notebook analyzes class imbalance, feature relationships, and pre-model feature importance for the US auto claims fraud dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

sns.set(style='whitegrid', context='notebook')

In [ ]:
data_root = Path('../data').resolve()
default_file = data_root / 'raw' / 'vehicle_insurance_fraud.csv'
if default_file.exists():
    df = pd.read_csv(default_file)
else:
    fallback = list((data_root / 'raw').glob('*.csv'))
    if not fallback:
        raise FileNotFoundError('No CSV found in ml_models/data/raw/')
    df = pd.read_csv(fallback[0])

df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fraud_counts = df['fraud_label'].value_counts().sort_index()
sns.countplot(x='fraud_label', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Fraud vs Non-Fraud Distribution')
axes[0].set_xlabel('fraud_label')
axes[0].set_ylabel('Count')

fraud_pct = fraud_counts / fraud_counts.sum() * 100
axes[1].pie(fraud_counts.values, labels=[f'Class {i}' for i in fraud_counts.index], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Imbalance Visualization')

plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=['number'])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
target_col = 'fraud_label'
X = df.drop(columns=[target_col])
y = df[target_col]

cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)
X_t = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_t, y)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
importance_df.head(20)

In [ ]:
top_n = 20
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df.head(top_n), x='importance', y='feature', palette='magma')
plt.title(f'Top {top_n} Feature Importances (Pre-model)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Notes
- The dataset is expected to be imbalanced (~90/10).
- Feature importance here is exploratory and model-dependent.
- Use `ml_models/data/preprocessing.py` and `imbalance.py` for production-ready preprocessing and balancing.